# Exploratory Data Analysis (EDA) - ADN Pipeline
Notebook này sử dụng kiến trúc OOP (`CocoEDAAnalyzer`) để trích xuất 6 biểu đồ phân tích chuyên sâu cho các tập dữ liệu rác thải (TACO và Roboflow).

In [ ]:
import os

os.chdir('/kaggle/working')
!rm -rf Detect-Waste-ADN
!git clone -b feature/hybrid-finetune https://github.com/nnnhuan251-hcmus/Detect-Waste-ADN.git
os.chdir('/kaggle/working/Detect-Waste-ADN')
!pip install -r requirements.txt
!pip install kagglehub roboflow
!pip install -e .
os.environ['PYTHONPATH'] = "src"
print("Xong bước 1!")

In [ ]:
import os
import shutil
import kagglehub
from roboflow import Roboflow

# Chắc chắn chúng ta đang đứng ở thư mục gốc dự án
os.chdir('/kaggle/working/Detect-Waste-ADN')

# Tạo sẵn thư mục chứa data cho chuẩn với kiến trúc
os.makedirs('data/raw/TACO', exist_ok=True)

print("⏳ Đang tải bộ dữ liệu 1 (Kaggle TACO)...")
# 1. Tải bộ dữ liệu 1 từ Kaggle
path_taco_cache = kagglehub.dataset_download("wilmacv251/taco-dataset-waste")

# Copy dữ liệu TACO vào đúng thư mục dự án
for item in os.listdir(path_taco_cache):
    s = os.path.join(path_taco_cache, item)
    d = os.path.join('data/raw/TACO', item)
    if os.path.isdir(s):
        shutil.copytree(s, d, dirs_exist_ok=True)
    else:
        shutil.copy2(s, d)

print("⏳ Đang tải bộ dữ liệu 2 (Roboflow)...")
# 2. Tải bộ dữ liệu 2 từ Roboflow thẳng vào thư mục data/raw
os.chdir('/kaggle/working/Detect-Waste-ADN/data/raw')
rf = Roboflow(api_key="5r1QbIwiFST8QcGoDpot")
project = rf.workspace("rf100-vl").project("taco-trash-annotations-in-context-dtyly-awiq")
version = project.version(2)
dataset = version.download("coco")

# Đổi tên thư mục Roboflow vừa tải về cho đúng tên gốc mà Notebook đang tìm kiếm
if os.path.exists(dataset.location):
    # Đổi tên thành thư mục "Roboflow"
    shutil.move(dataset.location, '/kaggle/working/Detect-Waste-ADN/data/raw/Roboflow')

# Quay lại thư mục gốc dự án
os.chdir('/kaggle/working/Detect-Waste-ADN')

print("✅ TUYỆT VỜI! Đã tải và tự động gom cả 2 bộ dữ liệu vào đúng vị trí!")

In [ ]:
import os
import sys
import json

# Khóa cứng đường dẫn gốc cho môi trường Kaggle
project_root = '/kaggle/working/Detect-Waste-ADN'

# Đảm bảo import được module từ src/
if project_root not in sys.path:
    sys.path.append(project_root)

from src.waste_detection.data.eda_analyzer import CocoEDAAnalyzer
print("Đã nạp công cụ thành công!")

## 1. Phân tích tập dữ liệu TACO (Đã ánh xạ 7 nhãn)
Khởi tạo Analyzer và cấu hình nhãn.

In [ ]:
# Đường dẫn tới dữ liệu TACO
taco_ann_path = os.path.join(project_root, 'data', 'raw', 'TACO', 'annotations.json')
taco_img_dir = os.path.join(project_root, 'data', 'raw', 'TACO')

# Đọc cấu hình ánh xạ từ 60 nhãn về 7 nhãn
mapping_file = os.path.join(project_root, 'configs', 'data', 'mapping_label.json')
with open(mapping_file, 'r', encoding='utf-8') as f:
    mapping_dict = json.load(f)

original_to_target = {}
for target_name, original_names in mapping_dict.items():
    for orig in original_names:
        original_to_target[orig] = target_name

def taco_mapper(cat_id, cat_name):
    mapped = original_to_target.get(cat_name)
    if mapped == "ignore" or mapped is None:
        return None
    return mapped

try:
    # Khởi tạo Analyzer
    taco_analyzer = CocoEDAAnalyzer(
        annotation_file=taco_ann_path,
        image_dir=taco_img_dir,
        dataset_name="TACO_7Class",
        label_mapper=taco_mapper
    )
    print("✅ Khởi tạo Analyzer thành công!")
except FileNotFoundError:
    print(f"❌ Không tìm thấy dữ liệu TACO tại {taco_ann_path}. Vui lòng tải dữ liệu về máy trước.")
    taco_analyzer = None

### 1.1 Phân bố số lượng rác theo từng lớp

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_class_distribution()

### 1.2 Phân bố kích thước Bounding Box (Width x Height)

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_bbox_size_distribution()

### 1.3 Histogram: Số lượng vật thể rác trên mỗi ảnh

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_objects_per_image()

### 1.4 Heatmap Không gian: Rác thường nằm ở đâu trong khung hình?

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_bbox_heatmap()

### 1.5 Tỉ lệ khung hình (Aspect Ratio W/H) - Quan trọng cho Anchor Box YOLO

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_aspect_ratio_distribution()

### 1.6 Ma trận đồng xuất hiện (Co-occurrence Matrix)

In [ ]:
if taco_analyzer:
    taco_analyzer.plot_class_cooccurrence()

---
## 2. Phân tích tập dữ liệu Roboflow (Nếu có)
Chỉ cần truyền đường dẫn của tập Roboflow vào bộ khởi tạo `CocoEDAAnalyzer` tương tự như trên.

In [ ]:
# Chèn thêm chữ 'train' vào đường dẫn
robo_ann_path = os.path.join(project_root, 'data', 'raw', 'Roboflow', 'train', '_annotations.coco.json')
robo_img_dir = os.path.join(project_root, 'data', 'raw', 'Roboflow', 'train')

try:
    robo_analyzer = CocoEDAAnalyzer(
        annotation_file=robo_ann_path,
        image_dir=robo_img_dir,
        dataset_name="Roboflow",
        label_mapper=taco_mapper  # Dùng chung công thức 7 nhãn với TACO
    )
    print("✅ Khởi tạo Analyzer cho Roboflow thành công!")
    # Gọi hàm vẽ biểu đồ cho Roboflow
    robo_analyzer.plot_class_distribution()
    robo_analyzer.plot_bbox_size_distribution()
    robo_analyzer.plot_objects_per_image()
    robo_analyzer.plot_bbox_heatmap()
    robo_analyzer.plot_aspect_ratio_distribution()
    robo_analyzer.plot_class_cooccurrence()
except FileNotFoundError:
    print(f"Thông báo: Chưa có dữ liệu Roboflow tại {robo_ann_path}. Bỏ qua bước này.")
    robo_analyzer = None